In [ ]:
from openai import OpenAI
import json
import os
from dotenv import load_dotenv
load_dotenv(override=True)
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),  # export OPENROUTER_API_KEY=...
)

def get_weather(city):
    return f"{city}: 18°C, lluvia ligera"

tools = [{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Obtiene el clima actual de una ciudad",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "Nombre de la ciudad"}
            },
            "required": ["city"]
        }
    }
}]

messages = [{"role": "user", "content": "¿Cómo está el clima en Quito?"}]

response = client.chat.completions.create(
    model="openai/gpt-5-mini",
    messages=messages,
    tools=tools,
)

msg = response.choices[0].message

if msg.tool_calls:
    messages.append(msg)
    for call in msg.tool_calls:
        args = json.loads(call.function.arguments)
        result = get_weather(args["city"])
        messages.append({
            "role": "tool",
            "tool_call_id": call.id,
            "content": result,
        })

    final = client.chat.completions.create(
        model="openai/gpt-5-mini",
        messages=messages,
        tools=tools,
    )
    print(final.choices[0].message.content)
else:
    print(msg.content)



En Quito está 18 °C con lluvia ligera. Te recomiendo llevar paraguas o impermeable y algo de abrigo ligero. ¿Quieres el pronóstico para las próximas horas o días?
